# XGBoost path — one asset end-to-end (NVDA)

The executed copy of the canonical per-asset template, run on the research snapshot of the
sealed final epoch (split-adjusted bars) by the research runner. It is thin orchestration over
the committed engine — this branch carries the same engine at `src/xgb/pipeline.py` +
`src/xgb/artifact.py` — so no logic is re-implemented here: what you read is what actually ran.

**The path (layers L4 → L9):**

| Layer | What happens |
|---|---|
| L4 | DuckDB bar snapshot → clean 1h parquet + deterministic 1d / 1w roll-ups (fail-closed source QC) |
| L5 | warmup / Train / OOS time split, with purge (= label horizon) and embargo at every boundary |
| L6 | causal entry candidates + asymmetric ATR Triple-Barrier labels → the "Output B" feature matrix |
| L7 | Optuna tunes XGBoost by **Train out-of-fold trading log-growth** at full-capital sizing |
| L8 | per-asset operating point (θ via shared `op_select`) + final fit on full Train → self-contained `strategy_NVDA.py` artifact (base64 booster + selfcheck) |
| L9 | the **ledgered OOS read** (2024 → 2026), HODL fallback on zero trades, one metrics row |

Honest framing: over the strong 2024→2026 bull window the strategy does **not** beat buy & hold
on most assets — the demonstrated product is the causal, OOS-isolated *method*. Prices are
split-adjusted from a reviewed event table before any roll-up, so the strategy and its
buy-and-hold benchmark read the same economics.

## 00 — Setup

In [1]:
import sys
from pathlib import Path

STRUCTURE_DIR = str(Path.cwd())
sys.path.insert(0, STRUCTURE_DIR)
import pipeline as P
import asset_writers as W

TICKER = "NVDA"
DB_PATH = P.bars_db()  # data/liora.duckdb (full) or data/bars_demo.duckdb (bundled demo)

ASSET_DIR = P.REPO_ROOT / "Assets" / TICKER
ASSET_DIR.mkdir(parents=True, exist_ok=True)
AF = {
    "notebook":   ASSET_DIR / f"{TICKER}__L4_to_L9.ipynb",
    "parquet":    ASSET_DIR / f"{TICKER}_ohlcv_1h.parquet",
    "parquet_1d": ASSET_DIR / f"{TICKER}_ohlcv_1d.parquet",
    "parquet_1w": ASSET_DIR / f"{TICKER}_ohlcv_1w.parquet",
    "optuna":     ASSET_DIR / "OPTUNAs_XGB_HPOs_best_params.json",
    "strategy":   ASSET_DIR / f"strategy_{TICKER}.py",
    "readme":     ASSET_DIR / f"{TICKER}_README.md",
}
P.validate_parameters()
SEED = P.seed_everything()
MANIFEST = P.resolve_feature_manifest(TICKER)
THRESH = P.PIPELINE_PARAMETERS["THRESHOLD_ENTRY"]
CAPITAL_MODE = P.PIPELINE_PARAMETERS["CAPITAL_MODE"]
print("TICKER", TICKER, "| DB", DB_PATH, "| features", MANIFEST["effective_feature_count"])
print("namespaces", MANIFEST["active_namespaces"])


TICKER NVDA | DB …/xgb/data/bars_demo.duckdb | features 17
namespaces ['1h']


## L4 — DuckDB snapshot → clean 1h parquet, then 1d / 1w roll-up

In [2]:
df = P.layer4_snapshot_to_parquet(DB_PATH, TICKER, AF["parquet"])
print("L4 clean 1h rows:", len(df), "|", df["timestamp"].min(), "->", df["timestamp"].max())


L4 clean 1h rows: 18209 | 2016-01-04 14:00:00+00:00 -> 2026-05-22 19:00:00+00:00


In [3]:
P.layer4_materialize_timeframes(df, {"1d": AF["parquet_1d"], "1w": AF["parquet_1w"]})
import pandas as pd
for tf in ("1d", "1w"):
    d = pd.read_parquet(AF[f"parquet_{tf}"])
    print(f"L4 {tf} rows:", len(d), "| cols:", list(d.columns))


L4 1d rows: 2612 | cols: ['timestamp', 'open', 'high', 'low', 'close', 'volume']
L4 1w rows: 542 | cols: ['timestamp', 'open', 'high', 'low', 'close', 'volume']


## L5 → L6 — time split, feature namespaces, Triple Barrier Y

In [4]:
REC = P.derive_output_b(df, TICKER, MANIFEST)
print("train candidates:", len(REC["train_events"]), "| Output-B rows:", len(REC["df_b"]))
print("bounds:", REC["bounds"])
print("audit:", REC["audit"])


…/xgb/src/pipeline.py:644: RuntimeWarning: Mean of empty slice
  out["momentum_alignment_multi"] = np.nanmean(momentum, axis=0)
…/xgb/src/pipeline.py:647: RuntimeWarning: Mean of empty slice
  out["macd_hist_alignment_multi"] = np.nanmean(macd, axis=0)
…/xgb/src/pipeline.py:652: RuntimeWarning: Mean of empty slice
  out["price_vs_sma_alignment_multi"] = np.nanmean(sma, axis=0)


train candidates: 9629 | Output-B rows: 9629
bounds: {'train_start_idx': 1393, 'train_end_idx': 14026, 'oos_start_idx': 14027, 'oos_end_idx': 18208}
audit: {'candidates': {'candidates': 9674}, 'eligibility': {'core_nan_excluded': 0, 'core_inf_excluded': 0}, 'scoring': {'train_core_ineligible_skipped': None, 'oos_core_ineligible_skipped': None}}


## L7 — Optuna best hyperparameters + Kelly

In [5]:
BEST, AP, FOLDS = P.layer7_optuna(REC["df"], REC["df_b"], REC["train_events"], REC["bounds"], SEED, MANIFEST)
print("best:", BEST)
print("cv AUC-PR:", AP, "| folds:", FOLDS)


…/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


best: {'max_depth': 4, 'eta': 0.2029276349470088, 'subsample': 0.7076744431409793, 'colsample_bytree': 0.6246975899880721, 'min_child_weight': 2.998640626673256, 'reg_lambda': 2.8866652956393883, 'reg_alpha': 1.9808542549562251, 'gamma': 0.17860724206352607, 'n_estimators': 122}
cv AUC-PR: 0.4280971669580838 | folds: 4


In [6]:
# min-2-trades mandate: the operating point is ALWAYS calibrated on Train-OOF under the
# trade floor (all-in: theta only, lambda=None) — a strategy must differ from HODL
CAL = P.calibrate_kelly(REC["df"], REC["df_b"], REC["train_events"], REC["bounds"], BEST, SEED, MANIFEST)
BEST["kelly_fraction"] = CAL["kelly_fraction"]
BEST["theta_entry"] = CAL["theta_entry"]
print("operating point: theta =", BEST["theta_entry"], "| kelly_fraction (lambda):", BEST["kelly_fraction"], "| CAPITAL_MODE:", CAPITAL_MODE)
print("calibration:", {k: CAL.get(k) for k in ("oof_trades", "trade_floor_met", "theta_boundary", "fold_spread_relaxed")})


operating point: theta = 0.4 | kelly_fraction (lambda): None | CAPITAL_MODE: all_in_compounding_per_asset
calibration: {'oof_trades': 844, 'trade_floor_met': True, 'theta_boundary': True, 'fold_spread_relaxed': False}


In [7]:
import json
json.dump({"best_params": BEST, "feature_manifest": MANIFEST,
           "features": {"ids": MANIFEST["effective_feature_ids"],
                        "names": MANIFEST["effective_feature_names"],
                        "active_namespaces": MANIFEST["active_namespaces"],
                        "per_namespace": MANIFEST["per_namespace"]},
           "cv_auc_pr": AP, "cv_folds": FOLDS, "n_trials": P.n_trials()},
          open(AF["optuna"], "w"), indent=2)
print("wrote", AF["optuna"].name)


wrote OPTUNAs_XGB_HPOs_best_params.json


## L8 — XGBoost strategy artifact

In [8]:
BOOSTER = P.layer8_train(REC["df_b"], BEST, SEED, MANIFEST)
KF = BEST.get("kelly_fraction") if CAPITAL_MODE.startswith("kelly") else None
tr_scored = P.score_setups(BOOSTER, REC["df"], REC["train_events"], MANIFEST, REC["feature_context"])
tr_summary, _, _ = P.run_engine(REC["df"], tr_scored, REC["bounds"]["train_start_idx"],
                                REC["bounds"]["oos_start_idx"] - 1, BEST.get("theta_entry", THRESH), kelly_fraction=KF)
ACCEPT = P.accept_strategy(tr_summary)
print("train acceptance:", ACCEPT)


train acceptance: {'accepted': True, 'mode': 'correctness_check', 'rankable': True, 'reason': 'MIN_TRAIN_ACCEPTANCE_TRADES is null -> correctness-check acceptance'}


In [9]:
sp = P.PIPELINE_PARAMETERS["splits"]
# lineage: the recipe hash this ticker's feature override was selected under (#8, #17)
_ov = P._load_per_asset_overrides().get(TICKER) or {}
LINEAGE = {"recipe_hash": (_ov.get("provenance") or {}).get("recipe_hash", "")}
META = P.strategy_meta(BOOSTER, REC["df_b"], TICKER, BEST, CAL, LINEAGE,
                       {"start": sp["train_start"], "end": sp["train_end"]}, MANIFEST)
META["ACCEPTANCE"] = ACCEPT
W.write_strategy(AF["strategy"], META)
print("wrote", AF["strategy"].name, "| artifact theta =", META["THRESHOLD_ENTRY"])


wrote strategy_NVDA.py | artifact theta = 0.4


## L9 — OOS verdict and endproduct

In [10]:
oos_events, _ = P.generate_candidate_events(df, REC["masks"]["oos"], REC["feature_context"])
oos_scored = P.score_setups(BOOSTER, df, oos_events, MANIFEST, REC["feature_context"])
SUMM, LEDGER, _ = P.run_engine(df, oos_scored, REC["bounds"]["oos_start_idx"],
                               REC["bounds"]["oos_end_idx"], BEST.get("theta_entry", THRESH), kelly_fraction=KF)
print("OOS:", {k: SUMM[k] for k in ("start_capital", "end_capital", "profit_factor", "trades")})

if SUMM["trades"] == 0:
    SUMM, LEDGER, _ = P.hodl_fallback(df, REC["bounds"]["oos_start_idx"], REC["bounds"]["oos_end_idx"])
    print("OOS model trades = 0 -> HODL fallback:",
          {k: SUMM[k] for k in ("start_capital", "end_capital", "return_pct", "trades")})

OOS: {'start_capital': 1000.0, 'end_capital': 3390.0761316752228, 'profit_factor': 1.301484675043824, 'trades': 288}


In [11]:
# result taxonomy + Train-OOF calibration record travel with the row (#2, #8; OOS only reports)
SUMM["result_mode"] = P.result_mode(SUMM["model_trades"], bool(CAL.get("trade_floor_met", False)))
SUMM["theta_entry"] = CAL["theta_entry"]
SUMM["oof_trades"] = CAL.get("oof_trades")
SUMM["trade_floor_met"] = CAL.get("trade_floor_met")
SUMM["oof_log_growth"] = CAL.get("oof_log_growth")
SUMM["recipe_hash"] = LINEAGE["recipe_hash"]
W.write_readme(AF["readme"], {**SUMM, "ticker": TICKER,
                              "capital_mode": SUMM.get("capital_mode", CAPITAL_MODE),
                              "kelly_fraction": None if SUMM.get("hodl_fallback") else BEST.get("kelly_fraction")},
               LEDGER, MANIFEST)
print("wrote", AF["readme"].name)

OOS_DB = Path(P.oos_db())  # data/oos_metrics.db (env OOS_METRICS_DB redirects it for make verify)
W.write_oos_metrics(OOS_DB, {**SUMM, "ticker": TICKER, "cv_auc_pr": AP, "cv_folds": FOLDS,
                             "oos_window": f'{sp["oos_start"]} -> {sp["oos_end"]}'})
print("L9 ->", OOS_DB.name, ":", TICKER, "|", SUMM["result_mode"])


wrote NVDA_README.md
L9 -> verify_scratch.db : NVDA | ML_MULTI_TRADE


In [12]:
six = [AF["parquet"], AF["parquet_1d"], AF["parquet_1w"], AF["optuna"], AF["strategy"], AF["readme"]]
missing = [p.name for p in six if not p.exists()]
assert not missing, f"missing deliverables: {missing}"
print("6/7 deliverables present (notebook copy added by run_asset.py):")
print(sorted(p.name for p in ASSET_DIR.iterdir() if p.is_file()))


6/7 deliverables present (notebook copy added by run_asset.py):
['NVDA_README.md', 'NVDA_ohlcv_1d.parquet', 'NVDA_ohlcv_1h.parquet', 'NVDA_ohlcv_1w.parquet', 'OPTUNAs_XGB_HPOs_best_params.json', 'strategy_NVDA.py']


---

**Reproducibility note.** The OOS verdict printed above is the sealed row this branch ships —
`data/results.db` → `asset_results` (NVDA, xgb): end capital **3390.0761…** (+239.0 %, 288 model
trades, profit factor 1.30) vs buy-and-hold **+336.6 %**. On the research branch the
`make verify-xgb` gate re-derives sample rows from committed data within a declared 1e-6
tolerance; on this branch the artifact tree is verified byte-for-byte against the SHA-256
hashes in `artifacts/manifest.json`.